Staging StatCan economic flows tables
=====================================

In [ ]:
from time import sleep
from pathlib import Path 
import pandas as pd 
import sys

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))
from utils.paths import RAW_STATCAN_DIR, STG_STATCAN_DIR

In [2]:
# Configuration
# - This dictionary drives the entire staging process.
# - It centralizes all the specific details for each raw file.

TRADE_IMPORTS_TABLE_ID = 12100115
TRADE_EXPORTS_TABLE_ID = 12100104
FDI_IMMEDIATE_TABLE_ID = 36100008
FDI_ULTIMATE_TABLE_ID  = 36100433

FILENAME_PREFIX = 'raw_statcan__tbl_'

STATCAN_CONFIG = {
    'imports': {
        'table_id': TRADE_IMPORTS_TABLE_ID,
        'cols_to_keep': ['REF_DATE', 'GEO', 'Country', 'Estimates', 'SCALAR_ID', 'VALUE'],
        'rename_map': {'REF_DATE': 'year', 'Country': 'imports_from_region', 'VALUE': 'dollar_value'},
        'category_col': 'Estimates',
        'category_val_to_keep': 'Value of imports',
    },
    'exports': {
        'table_id': TRADE_EXPORTS_TABLE_ID,
        'cols_to_keep': ['REF_DATE',
                         'GEO',
                         'Country of destination',
                         'Estimates',
                         'SCALAR_ID',
                         'VALUE'],
        'rename_map': {'REF_DATE': 'year', 'Country of destination': 'exports_to_region', 'VALUE': 'dollar_value'},
        'category_col': 'Estimates',
        'category_val_to_keep': 'Value of exports',
    },
    'fdi_immediate': {
        'table_id': FDI_IMMEDIATE_TABLE_ID,
        'cols_to_keep': ['REF_DATE',
                         'GEO',
                         'Canadian and foreign direct investment',
                         'Countries or regions',
                         'SCALAR_ID',
                         'VALUE'],
        'rename_map': {
            'REF_DATE': 'year',
            'Canadian and foreign direct investment': 'fdi_type',
            'Countries or regions': 'region',
            'VALUE': 'dollar_value'
        },
        'category_col': 'Canadian and foreign direct investment',
        'category_val_to_keep': [
            'Canadian direct investment abroad - total book value',
            'Foreign direct investment in Canada - total book value'
        ],
    },
    'fdi_ultimate': {
        'table_id': FDI_ULTIMATE_TABLE_ID,
        'cols_to_keep': ['REF_DATE',
                         'GEO',
                         'Type of direct investment',
                         'Countries or regions',
                         'SCALAR_ID',
                         'VALUE'],
        'rename_map': {
            'REF_DATE': 'year',
            'Type of direct investment': 'fdi_type',
            'Countries or regions': 'region',
            'VALUE': 'dollar_value'
        },
        'category_col': 'Type of direct investment',
        'category_val_to_keep': [
            'Canadian direct investment abroad by ultimate investor country – total book value',
            'Foreign direct investment in Canada by ultimate investor country - total book value'
        ],
    },
}

In [3]:
def stage_statcan_data(table_id: int, config: dict) -> pd.DataFrame:
    print(f'Staging table [{table_id}]... ', end='')
    
    # Load data
    filepath = RAW_STATCAN_DIR / f'{FILENAME_PREFIX}{table_id}.csv'
    df = pd.read_csv(filepath, usecols=config['cols_to_keep'])
    
    # Filter rows
    df = df[df['GEO'] == 'Canada'].copy()
    
    # Keep relevant categories
    categories_to_keep: str | list = config['category_val_to_keep']
    if not isinstance(categories_to_keep, list):
        categories_to_keep = [categories_to_keep]
    
    # Filter for desired categories in category column
    df: pd.DataFrame = (df[df[config['category_col']].isin(categories_to_keep)]
                        .copy())
    
    # Remove scaling from value column
    df['VALUE']     = pd.to_numeric(df['VALUE'], errors='coerce')
    df['SCALAR_ID'] = pd.to_numeric(df['SCALAR_ID'], errors='coerce')

    df['VALUE'] = df['VALUE'] * (10 ** df['SCALAR_ID'])
    
    # Rename and select final columns
    df = df.rename(columns=config['rename_map'])
    cols_to_keep = list(config['rename_map'].values())
    df = df[cols_to_keep]
    
    # Cast column datatypes
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int32')
    df['dollar_value'] = df['dollar_value'].astype('Float64')
    
    print(f'Staging complete. Shape: {df.shape}') 
    
    return df

In [4]:
for name, config in STATCAN_CONFIG.items():
    # Get table id and stage StatCan table
    table_id = config['table_id']
    stg_df = stage_statcan_data(table_id, config)
    
    # Save staged StatCan data
    stg_filename = f'stg_statcan__{name}.csv'
    stg_filepath = STG_STATCAN_DIR / stg_filename
    stg_df.to_csv(stg_filepath, index=False)
    
    print(f'Staged file saved to {stg_filepath}\n')    

Staging table [12100115]... Staging complete. Shape: (4343, 3)
Staged file saved to C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\statcan\stg_statcan__imports.csv

Staging table [12100104]... Staging complete. Shape: (5890, 3)
Staged file saved to C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\statcan\stg_statcan__exports.csv

Staging table [36100008]... Staging complete. Shape: (13533, 4)
Staged file saved to C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\statcan\stg_statcan__fdi_immediate.csv

Staging table [36100433]... Staging complete. Shape: (863, 4)
Staged file saved to C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\statcan\stg_statcan__fdi_ultimate.csv

